In [21]:
# Cell 1: Import core libraries and project modules
import torch
import torch.nn as nn
import torch.optim as optim
import wandb

# Import modular components directly from the src package
from src.dataset import get_data_loaders
from src.models import TinyCNN
from src.utils import run_sanity_check

# Authenticate with Weights & Biases for experiment tracking
wandb.login()

True

In [22]:
# Cell 2: Execute sanity checks for Architecture 1 (TinyCNN)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Run the automated forward/backward and 1-batch overfit checks
run_sanity_check(TinyCNN, device)

--- Running Sanity Check for TinyCNN ---
-> Forward Pass Verification: PASSED
-> Backward Gradient Computation: PASSED
-> 1-Batch Overfit Test: Initial Loss: 1.8251 -> Final Loss: 0.0000
--- Sanity Check Complete ---


In [29]:
# Cell 5: Run the final optimized experiment with BatchNorm, Dropout, and Augmentation
wandb.init(project="facial-expression-recognition", name="03_optimized_final", config={
    "architecture": "OptimizedCNN",
    "learning_rate": 0.001,
    "epochs": 15,
    "batch_size": 2,
    "dropout_rate": 0.4
})

config = wandb.config

# Import the final optimized architecture
from src.models import OptimizedCNN

# Run mandatory verification checks for the optimized architecture
print("--- Running Sanity Check for OptimizedCNN ---")
run_sanity_check(OptimizedCNN, device)
print("---------------------------------------------")

# Activate data augmentation to increase generalization power
train_loader, val_loader, _ = get_data_loaders(batch_size=config.batch_size, use_augmentation=True)

# Initialize optimized architecture with configured dropout probability
model = OptimizedCNN(dropout_rate=config.dropout_rate).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

# Run the definitive training sequence
for epoch in range(1, config.epochs + 1):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)

    train_loss, train_acc = total_loss / total, correct / total

    # Run evaluation track on the validation split
    model.eval()
    v_loss, v_correct, v_total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            v_loss += loss.item() * imgs.size(0)
            v_correct += (outputs.argmax(1) == labels).sum().item()
            v_total += imgs.size(0)
    val_loss, val_acc = v_loss / v_total, v_correct / v_total

    # Print metrics on screen and stream them directly to W&B
    print(f"Epoch {epoch:02d} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%")
    wandb.log({"epoch": epoch, "train/loss": train_loss, "train/accuracy": train_acc, "val/loss": val_loss, "val/accuracy": val_acc})

# Formally close out the tracking pipeline
wandb.finish()

--- Running Sanity Check for OptimizedCNN ---
--- Running Sanity Check for OptimizedCNN ---
-> Forward Pass Verification: PASSED
-> Backward Gradient Computation: PASSED
-> 1-Batch Overfit Test: Initial Loss: 2.3235 -> Final Loss: 0.0004
--- Sanity Check Complete ---
---------------------------------------------
Epoch 01 | Train Acc: 2.94% | Val Acc: 14.29%
Epoch 02 | Train Acc: 20.59% | Val Acc: 14.29%
Epoch 03 | Train Acc: 23.53% | Val Acc: 14.29%
Epoch 04 | Train Acc: 11.76% | Val Acc: 14.29%
Epoch 05 | Train Acc: 8.82% | Val Acc: 14.29%
Epoch 06 | Train Acc: 14.71% | Val Acc: 14.29%
Epoch 07 | Train Acc: 14.71% | Val Acc: 14.29%
Epoch 08 | Train Acc: 11.76% | Val Acc: 14.29%
Epoch 09 | Train Acc: 26.47% | Val Acc: 14.29%
Epoch 10 | Train Acc: 8.82% | Val Acc: 14.29%
Epoch 11 | Train Acc: 17.65% | Val Acc: 14.29%
Epoch 12 | Train Acc: 20.59% | Val Acc: 14.29%
Epoch 13 | Train Acc: 8.82% | Val Acc: 14.29%
Epoch 14 | Train Acc: 14.71% | Val Acc: 14.29%
Epoch 15 | Train Acc: 11.76% | V

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train/accuracy,▁▆▇▄▃▄▄▄█▃▅▆▃▄▄
train/loss,█▄▃▂▃▄▃▄▁▃▄▂▃▂▃
val/accuracy,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/loss,█▆▄▂▂▂▅▂▄▆▃▅▁▁▃
epoch,15
train/accuracy,0.11765
train/loss,2.05827
val/accuracy,0.14286
val/loss,1.97934
